# Figure 11 — get the noise wrong and the outcome becomes a lottery

Two BO campaigns, same landscape, same seeds, same true measurement noise (sd = 6 percentage points). One is told the noise is ~0, the other is told the truth.

**Two honest findings.** First, under noise the best *observed* yield overstates what those conditions really give — for both settings, because the maximum of noisy readings is biased upwards. Second, and this is the one that matters: telling the GP sigma_n ~ 0 does not just cost a couple of points on average, it makes the result *wildly variable*. The same consistency argument as the human benchmark.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

## Setup

28 random seeds each, 6 seed experiments, 14 BO iterations. Per run we record what a chemist would report (highest observed yield), what those conditions are actually worth, and what the *model* would recommend (argmax of the posterior mean).

In [ ]:

style.SHOW_TEXT = True   # False -> clean panels (no titles/captions/labels)
                         # for ungrouping the PDF into editable pptx shapes

NOISE = 6.0
N_SEEDS = 28
N_ITER = 14
gx = np.linspace(*land.BOUNDS[0], 36)
gy = np.linspace(*land.BOUNDS[1], 36)
cand = np.array([[a, b] for a in gx for b in gy])
_, zstar = land.optimum()
LS, SF = [20.0, 1.2], 28.0

res = {}
for tag, assumed in [(r"told $\sigma_n \approx 0$", 0.05),
                     (r"told the measured $\sigma_n$", NOISE)]:
    rep, true_at_rep, pick = [], [], []
    for s in range(N_SEEDS):
        r = gpmod.bo_loop(land.f, land.BOUNDS, n_init=6, n_iter=N_ITER, acq="ei",
                          noise=NOISE, assumed_sn=assumed, seed=s, grid=cand,
                          ls=LS, sf=SF)
        i = int(np.argmax(r["y"]))
        rep.append(r["y"][i]); true_at_rep.append(r["ytrue"][i])
        g = gpmod.GP(gpmod.matern52, ls=LS, sf=SF,
                     sn=max(assumed, 1e-4)).fit(r["X"], r["y"])
        mu, _ = g.predict(cand)
        pick.append(land.f(cand[int(np.argmax(mu))]))
    res[tag] = {k: np.array(v) for k, v in
                dict(rep=rep, true=true_at_rep, pick=pick).items()}
    d = res[tag]
    print(f"{tag:32s} reported {np.median(d['rep']):5.1f} | actually worth "
          f"{np.median(d['true']):5.1f} | optimism gap {np.median(d['rep']-d['true']):4.1f}")
    print(f"{'':32s} model's pick: median {np.median(d['pick']):5.1f}  "
          f"mean {d['pick'].mean():5.1f}  sd {d['pick'].std():4.1f}  "
          f"worst {d['pick'].min():5.1f}")
print("true optimum:", round(zstar, 1))

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(style.FIG_W_FULL, 3.6),
                         gridspec_kw=dict(wspace=0.26, width_ratios=[1.1, 1]))
cols = [style.RED, style.TEAL]
rng = np.random.default_rng(0)

# ---- left: distribution of what the model actually recommends -----------------
ax = axes[0]
for gi, ((tag, d), c) in enumerate(zip(res.items(), cols)):
    v = d["pick"]
    ax.scatter(gi + rng.uniform(-0.11, 0.11, v.size), v, s=26, color=c,
               alpha=0.55, edgecolor="none", zorder=4)
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    ax.add_patch(plt.Rectangle((gi - 0.22, q1), 0.44, q3 - q1, fc="none",
                              ec=c, lw=1.4, zorder=5))
    ax.plot([gi - 0.22, gi + 0.22], [med, med], color=c, lw=2.6, zorder=6)
    style.text(ax, gi + 0.30, med, f"median {med:.0f}%", fontsize=9, color=c,
               va="center", fontweight="bold")
    style.text(ax, gi, 36.0, f"sd {v.std():.1f}   worst {v.min():.0f}%", ha="center",
               fontsize=10.0, color=c, fontweight="bold",
               bbox=dict(fc="white", alpha=0.85, ec="none", pad=2.0))
ax.axhline(zstar, color=style.INK, ls=":", lw=1.0)
style.text(ax, 1.45, zstar + 1.2, "true optimum", ha="right", fontsize=9,
           color=style.INK)
ax.set_xticks([0, 1]); ax.set_xticklabels(list(res), fontsize=11)
ax.set_xlim(-0.45, 1.55)
ax.set_ylim(33, 100)
style.ylabel(ax, "true yield at the model's recommended conditions / %",
             fontsize=9.5)
style.title(ax, "what the model ends up recommending", loc="left",
            fontsize=11.0, pad=18)
style.text(ax, 0.5, 1.005, "one dot = one campaign", transform=ax.transAxes,
           ha="center", fontsize=9.0, color=style.GRAY, va="bottom")

# ---- right: the optimism gap, which is there either way -----------------------
ax = axes[1]
W = 0.30
for gi, ((tag, d), c) in enumerate(zip(res.items(), cols)):
    for ki, (k, al, lab) in enumerate([("rep", 0.85, "reported"),
                                       ("true", 0.35, "actually worth")]):
        x = gi + (ki - 0.5) * W
        med = np.median(d[k])
        q1, q3 = np.percentile(d[k], [25, 75])
        ax.bar(x, med, width=W * 0.86, color=c, alpha=al, edgecolor=c, lw=1.0)
        ax.errorbar(x, med, yerr=[[med - q1], [q3 - med]], fmt="none",
                    ecolor=style.INK, elinewidth=1.0, capsize=3, alpha=0.65)
        style.text(ax, x, med + 0.8, f"{med:.0f}", ha="center", fontsize=9,
                   color=style.INK)
    gap = np.median(d["rep"] - d["true"])
    style.annotate(ax, f"overstated by {gap:.1f}", (gi, 99.5), ha="center",
                   fontsize=9.5, color=c, fontweight="bold")
ax.axhline(zstar, color=style.INK, ls=":", lw=1.0)
ax.set_xticks([0, 1]); ax.set_xticklabels(list(res), fontsize=11)
ax.set_ylim(70, 104)
style.ylabel(ax, "yield / %")
style.title(ax, "your reported best overstates reality — either way", loc="left",
            fontsize=10.5)
hand = [plt.Rectangle((0, 0), 1, 1, fc=style.GRAY, alpha=a_, ec=style.GRAY)
        for a_ in (0.85, 0.35)]
style.legend(ax, hand, ["highest observed yield", "what it is actually worth"],
             fontsize=8.6, loc="upper center", bbox_to_anchor=(0.5, -0.10),
             ncol=2, handlelength=1.1)
style.save(fig, "fig_11_noise_ablation", OUT)